# 02b — Ablaciones E1–E5 (registro histórico)

> Este notebook es el **registro** de la ablación (losses y etiquetas débiles). Sus resultados ya están en Drive. El camino final está en `02_modelado.ipynb`.

## Fase 2: Dataset, U-Net y modelo base

Objetivo de esta fase:
1. Cargar los chips en memoria con la máscara correcta (excluyendo lo que no sirve).
2. Normalizar el radar con estadísticas **solo de train**.
3. Definir la U-Net, la loss y las métricas **ignorando los píxeles −1**.
4. Medir la línea base clásica (umbral Otsu).
5. Entrenar **E1: U-Net + BCE** y compararla con Otsu en `valid`.

> El conjunto `test` y Bolivia **no se tocan** en esta fase. Solo se usan al final con el mejor modelo.

### Decisiones que salen de la exploración (`inventario_chips.csv`)

| Hallazgo | Decisión |
|---|---|
| 6 chips de Ghana tienen ≥ 99 % de píxeles −1, y el chip `Paraguay_34417` tiene la imagen 100 % NaN | Se **excluyen** (7 chips). Quedan 250 / 86 / 88 / 15 |
| Hay máscaras con etiqueta donde la imagen es NaN (sin medición de radar) | Esos píxeles pasan a **−1**: no se puede predecir sin imagen |
| Solo ≈ 10.6 % de los píxeles es agua; mediana 2.5 % por chip; 52 chips sin agua | Métricas de la **clase agua** (IoU, F1), no accuracy |
| Diferencia agua–tierra en VV: mediana 5.2 dB; en 99 chips < 3 dB | Un umbral fijo no alcanza → justifica la red (y la comparamos con Otsu) |

## 0. Preparación

In [ ]:
!pip install -q rasterio

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, random
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

ROOT = '/content/drive/MyDrive/tarea2_floods'
DATA = f'{ROOT}/data/sen1floods11'
OUT  = f'{ROOT}/outputs'
os.makedirs(OUT, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else '⚠️ activa la GPU: Entorno de ejecución → Cambiar tipo → T4')

La línea base necesita las máscaras del umbral Otsu (≈ 3 MB). Se descargan una sola vez.

In [ ]:
if not os.path.exists(f'{DATA}/S1OtsuLabelHand'):
    !gsutil -m -q cp -r gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1OtsuLabelHand "{DATA}/"
print('S1OtsuLabelHand:', len(os.listdir(f'{DATA}/S1OtsuLabelHand')), 'archivos (esperado 446)')

## 1. Lista de chips y exclusiones

Unimos las particiones con el inventario de la exploración y marcamos los chips inservibles.

In [ ]:
inv = pd.read_csv(f'{DATA}/inventario_chips.csv')
inv['chip'] = inv['evento'] + '_' + inv['chip_id'].astype(str)

parts = []
for s in ['train', 'valid', 'test', 'bolivia']:
    df = pd.read_csv(f'{DATA}/splits/flood_{s}_data.csv', header=None, names=['s1_file', 'label_file'])
    df['split'] = s
    parts.append(df)
chips = pd.concat(parts, ignore_index=True)
chips['chip'] = chips['s1_file'].str.replace('_S1Hand.tif', '', regex=False)
chips = chips.merge(inv[['chip', 'evento', 'pct_agua', 'pct_sin_dato', 'pct_nan']], on='chip', how='left')

chips['excluir'] = (chips['pct_sin_dato'] >= 99) | (chips['pct_nan'] >= 99)
chips[chips['excluir']]

In [ ]:
chips = chips[~chips['excluir']].reset_index(drop=True)
chips['split'].value_counts()      # esperado: train 250, test 88, valid 86, bolivia 15

## 2. Cargar los chips en memoria

Leer 400+ archivos desde Drive en cada época sería lento. Los leemos **una vez** y los guardamos en un diccionario (`CACHE`): ≈ 1 GB de RAM.

`read_chip` devuelve la imagen cruda en dB (con NaN) y la máscara con −1 también donde la imagen no tiene dato.

In [ ]:
def read_chip(img_path, label_path):
    """Imagen cruda (2, 512, 512) en dB con NaN, y máscara int8 con -1 / 0 / 1."""
    with rasterio.open(img_path) as src:
        img = src.read().astype('float32')
    with rasterio.open(label_path) as src:
        lab = src.read(1).astype('int8')
    sin_imagen = np.isnan(img).any(axis=0)
    lab[sin_imagen] = -1          # sin medición de radar → ese píxel no se evalúa
    return img, lab


CACHE = {}
for r in tqdm(chips.itertuples(), total=len(chips)):
    CACHE[r.chip] = read_chip(f'{DATA}/S1Hand/{r.s1_file}', f'{DATA}/LabelHand/{r.label_file}')
print('Chips en memoria:', len(CACHE))

In [ ]:
WEAK = '/content/weak'      # chips débiles: se descargan en la sección 14 al disco local de Colab (no a Drive)

def get_chip(chip):
    """Dorados: desde la RAM. Débiles: se leen del disco local cada vez (son 4,000+, no caben en RAM)."""
    if chip in CACHE:
        return CACHE[chip]
    return read_chip(f'{WEAK}/S1Weak/{chip}_S1Weak.tif', f'{WEAK}/S1OtsuLabelWeak/{chip}_S1OtsuLabelWeak.tif')

## 3. Normalización (solo con train)

La red aprende mejor con valores alrededor de 0. Pasos:
1. **Recortar extremos** (percentiles 0.1 % y 99.9 %): unos pocos píxeles muy brillantes (ciudades) o muy oscuros no deben dominar.
2. **Estandarizar**: `(x − media) / desviación` por banda.
3. Los NaN pasan a 0 (y en la máscara ya son −1, así que no cuentan).

Las estadísticas salen **solo de train** (una muestra de 20,000 píxeles por chip). Usar valid/test aquí sería "espiar" las respuestas.

In [ ]:
rng = np.random.default_rng(SEED)
muestras = []
for chip in chips.loc[chips['split'] == 'train', 'chip']:
    img, _ = CACHE[chip]
    px = img.reshape(2, -1)
    px = px[:, ~np.isnan(px).any(axis=0)]
    idx = rng.choice(px.shape[1], size=min(20000, px.shape[1]), replace=False)
    muestras.append(px[:, idx])
muestras = np.concatenate(muestras, axis=1)

pd.DataFrame(muestras.T, columns=['VV', 'VH']).describe(percentiles=[.001, .01, .5, .99, .999]).round(2)

In [ ]:
LO = np.percentile(muestras, 0.1, axis=1)
HI = np.percentile(muestras, 99.9, axis=1)
recortado = np.clip(muestras, LO[:, None], HI[:, None])
STATS = {'lo': LO.tolist(), 'hi': HI.tolist(),
         'mean': recortado.mean(axis=1).tolist(), 'std': recortado.std(axis=1).tolist()}
json.dump(STATS, open(f'{OUT}/norm_stats.json', 'w'), indent=2)    # se reutiliza para Lima y Bolivia

pd.DataFrame(STATS, index=['VV', 'VH']).round(2)

In [ ]:
def normalize(img):
    """dB crudo → recorta extremos, estandariza por banda y pone 0 donde no hay dato."""
    lo, hi = np.array(STATS['lo'])[:, None, None], np.array(STATS['hi'])[:, None, None]
    mean, std = np.array(STATS['mean'])[:, None, None], np.array(STATS['std'])[:, None, None]
    x = (np.clip(img, lo, hi) - mean) / std
    return np.nan_to_num(x, nan=0.0).astype('float32')

## 4. Dataset y aumentaciones

- **Train:** espejos y rotaciones de 90° al azar (el agua no tiene "arriba" ni "abajo") + **recorte aleatorio de 256×256**. El recorte da más variedad por época y entrena ≈ 4 veces más rápido.
- **Valid / test:** chip completo de 512×512, sin aumentaciones (la U-Net acepta cualquier tamaño múltiplo de 16).
- Sin cambios de brillo ni color: en radar el brillo **es** la información.

In [ ]:
def augment(img, lab, rng):
    if rng.random() < 0.5:
        img, lab = img[:, :, ::-1], lab[:, ::-1]                     # espejo horizontal
    if rng.random() < 0.5:
        img, lab = img[:, ::-1, :], lab[::-1, :]                     # espejo vertical
    k = int(rng.integers(4))
    return np.rot90(img, k, axes=(1, 2)), np.rot90(lab, k)          # rotación 0/90/180/270°


def random_crop(img, lab, size, rng):
    H, W = lab.shape
    y, x = int(rng.integers(0, H - size + 1)), int(rng.integers(0, W - size + 1))
    return img[:, y:y + size, x:x + size], lab[y:y + size, x:x + size]


class FloodDataset(Dataset):
    def __init__(self, df, train=False, crop=None):
        self.chips = df['chip'].tolist() if isinstance(df, pd.DataFrame) else list(df)    # DataFrame o lista de nombres
        self.train = train
        self.crop = crop

    def __len__(self):
        return len(self.chips)

    def __getitem__(self, i):
        img, lab = get_chip(self.chips[i])
        img = normalize(img)
        if self.train:
            rng = np.random.default_rng(int(torch.randint(2**31, (1,))))    # semilla desde torch → reproducible
            img, lab = augment(img, lab, rng)
            if self.crop:
                img, lab = random_crop(img, lab, self.crop, rng)
        img = torch.from_numpy(np.ascontiguousarray(img))
        lab = torch.from_numpy(np.ascontiguousarray(lab).astype('int64'))
        return img, lab

Revisamos qué entrega el Dataset: imagen `(2, 256, 256)` float32 normalizada y máscara `(256, 256)` con −1 / 0 / 1.

In [ ]:
ds_train = FloodDataset(chips[chips['split'] == 'train'], train=True, crop=256)
x, y = ds_train[0]
print('Imagen :', tuple(x.shape), x.dtype)
print('Máscara:', tuple(y.shape), y.dtype, '| valores:', torch.unique(y).tolist())
pd.DataFrame(x.reshape(2, -1).T.numpy(), columns=['VV', 'VH']).describe().round(2)    # media ≈ 0, desviación ≈ 1

In [ ]:
def show_mask(ax, lab, title):
    rgb = np.zeros((*lab.shape, 3))
    rgb[lab == 0] = [0.89, 0.84, 0.72]      # no agua (beige)
    rgb[lab == 1] = [0.12, 0.44, 0.70]      # agua (azul)
    rgb[lab == -1] = [0.60, 0.60, 0.60]     # sin dato (gris)
    ax.imshow(rgb); ax.set_title(title); ax.axis('off')


ejemplos = chips[chips['split'] == 'train'].sample(3, random_state=SEED)['chip']
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for row, chip in zip(axes, ejemplos):
    img, lab = CACHE[chip]
    row[0].imshow(img[0], cmap='gray', vmin=-25, vmax=0);  row[0].set_title(f'{chip} · VV'); row[0].axis('off')
    row[1].imshow(img[1], cmap='gray', vmin=-30, vmax=-5); row[1].set_title('VH'); row[1].axis('off')
    show_mask(row[2], lab, 'Máscara (azul agua · gris sin dato)')
plt.tight_layout(); plt.show()

## 5. U-Net (implementación propia)

Misma idea que la U-Net de fútbol, adaptada:
- **Entrada: 2 canales** (VV, VH) en vez de 3 (RGB).
- **Salida: 1 canal** (logit de "agua"); `logit > 0` ⇔ probabilidad > 0.5.
- Filtros `(32, 64, 128, 256)`: la mitad que en fútbol → ≈ 7.8 M parámetros. Con 250 chips, una red más grande memoriza más fácil.

| Paso | Forma con entrada 256×256 |
|---|---|
| Entrada | (B, 2, 256, 256) |
| Bajada 1–4 (+ guarda skip) | 32 @ 256 → 64 @ 128 → 128 @ 64 → 256 @ 32 |
| Fondo (bottleneck) | 512 @ 16 |
| Subida 4–1 (+ concatena skip) | 256 @ 32 → 128 @ 64 → 64 @ 128 → 32 @ 256 |
| Cabeza 1×1 | (B, 1, 256, 256) |

In [ ]:
def double_conv(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
    )


class UNet(nn.Module):
    def __init__(self, in_channels=2, out_channels=1, features=(32, 64, 128, 256)):
        super().__init__()
        self.down = nn.ModuleList()
        ch = in_channels
        for f in features:
            self.down.append(double_conv(ch, f))
            ch = f
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = double_conv(ch, ch * 2)
        self.up = nn.ModuleList()
        self.dec = nn.ModuleList()
        for f in reversed(features):
            self.up.append(nn.ConvTranspose2d(f * 2, f, kernel_size=2, stride=2))
            self.dec.append(double_conv(f * 2, f))
        self.head = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skips = []
        for block in self.down:
            x = block(x)
            skips.append(x)
            x = self.pool(x)
        x = self.bottleneck(x)
        for up, dec, skip in zip(self.up, self.dec, reversed(skips)):
            x = up(x)
            x = torch.cat([skip, x], dim=1)
            x = dec(x)
        return self.head(x)      # logits (B, 1, H, W)

In [ ]:
model = UNet().to(DEVICE)
salida = model(torch.randn(2, 2, 256, 256, device=DEVICE))
print('Entrada: (2, 2, 256, 256) → Salida:', tuple(salida.shape))
print(f'Parámetros: {sum(p.numel() for p in model.parameters()):,}')

## 6. Funciones de pérdida con máscara

Las tres losses del estudio de ablación. Todas **ignoran los píxeles −1**: se multiplican por `valid` (1 si el píxel cuenta, 0 si no).

| Loss | Qué mide | Fortaleza |
|---|---|---|
| `bce` | Error píxel a píxel (entropía cruzada binaria) | Estable, estándar |
| `dice` | Solapamiento entre la mancha predicha y la real | No se deja dominar por el 90 % de "no agua" |
| `bce_dice` | 0.5 · BCE + 0.5 · Dice | Combina ambas |

In [ ]:
def masked_bce(logits, target):
    valid = (target != -1).float()
    t = target.clamp(min=0).float()
    loss = F.binary_cross_entropy_with_logits(logits.squeeze(1), t, reduction='none')
    return (loss * valid).sum() / valid.sum().clamp(min=1)


def masked_dice(logits, target, eps=1.0):
    valid = (target != -1).float()
    p = torch.sigmoid(logits.squeeze(1)) * valid
    t = target.clamp(min=0).float() * valid
    inter = (p * t).sum()
    return 1 - (2 * inter + eps) / (p.sum() + t.sum() + eps)


LOSSES = {
    'bce': masked_bce,
    'dice': masked_dice,
    'bce_dice': lambda logits, target: 0.5 * masked_bce(logits, target) + 0.5 * masked_dice(logits, target),
}

**Prueba de la máscara:** si solo cambiamos la predicción en un píxel −1, la loss no debe cambiar.

In [ ]:
target   = torch.tensor([[[1, 0, -1]]])               # 3 píxeles: agua, no agua, sin dato
logits_a = torch.tensor([[[[2.0, -2.0,  5.0]]]])
logits_b = torch.tensor([[[[2.0, -2.0, -5.0]]]])      # solo cambia el píxel sin dato
for name, fn in LOSSES.items():
    print(f'{name:9s} a = {fn(logits_a, target):.4f}   b = {fn(logits_b, target):.4f}   ← deben ser iguales')

## 7. Métricas de la clase agua

Se acumula la matriz de confusión sobre **todos los píxeles válidos** de un split y al final se calculan:

| Métrica | Fórmula | Pregunta que responde |
|---|---|---|
| IoU | TP / (TP + FP + FN) | ¿Cuánto se solapan la mancha predicha y la real? (métrica principal) |
| Precisión | TP / (TP + FP) | De lo que marqué como agua, ¿cuánto era agua? |
| Recall | TP / (TP + FN) | Del agua real, ¿cuánta encontré? |
| F1 | 2·P·R / (P + R) | Balance entre precisión y recall |
| Omisión | FN / (TP + FN) | Agua real que no detecté (= 1 − recall) |
| Comisión | FP / (TP + FP) | Falsa agua (= 1 − precisión) |

In [ ]:
class WaterMetrics:
    """Matriz de confusión de la clase agua sobre los píxeles válidos (target != -1)."""
    def __init__(self):
        self.tp = self.fp = self.fn = self.tn = 0

    def update(self, pred, target):
        """pred: array bool (agua sí/no); target: array con -1 / 0 / 1 del mismo tamaño."""
        valid, agua = target != -1, target == 1
        self.tp += int(( pred &  agua & valid).sum())
        self.fp += int(( pred & ~agua & valid).sum())
        self.fn += int((~pred &  agua & valid).sum())
        self.tn += int((~pred & ~agua & valid).sum())

    def compute(self):
        tp, fp, fn, tn = self.tp, self.fp, self.fn, self.tn
        precision, recall = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
        return {
            'iou': tp / max(tp + fp + fn, 1),
            'f1': 2 * precision * recall / max(precision + recall, 1e-9),
            'precision': precision,
            'recall': recall,
            'omision': fn / max(tp + fn, 1),
            'comision': fp / max(tp + fp, 1),
            'accuracy': (tp + tn) / max(tp + fp + fn + tn, 1),
        }

    def confusion(self):
        return pd.DataFrame([[self.tn, self.fp], [self.fn, self.tp]],
                            index=['real: no agua', 'real: agua'], columns=['pred: no agua', 'pred: agua'])

## 8. Línea base: umbral de Otsu

`S1OtsuLabelHand` es lo que da el método clásico sobre los mismos chips. Es la vara mínima: **la U-Net debe superarla**.

In [ ]:
def eval_otsu(split):
    m = WaterMetrics()
    for chip in chips.loc[chips['split'] == split, 'chip']:
        with rasterio.open(f'{DATA}/S1OtsuLabelHand/{chip}_S1OtsuLabelHand.tif') as src:
            otsu = src.read(1)
        _, lab = CACHE[chip]
        m.update(otsu == 1, lab)
    return m


otsu_valid = eval_otsu('valid')
pd.Series(otsu_valid.compute()).round(4).to_frame('Otsu · valid')

In [ ]:
otsu_valid.confusion()

## 9. Entrenamiento

- `train_one_epoch`: una pasada por train con *mixed precision* (más rápido en la T4).
- `evaluate`: loss + métricas en un split, sin aprender.
- `run_experiment`: entrena, evalúa en valid cada época, guarda el **mejor checkpoint por IoU** y el historial en `outputs/<nombre>/`.

Así cada experimento de la ablación es una sola línea.

**Si el experimento ya está en Drive, no se vuelve a entrenar**: se lee su `history.csv`. Por eso se puede usar "Ejecutar todo" en cada sesión sin perder tiempo. Para reentrenar uno a propósito se pasa `force=True`.

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn, scaler):
    model.train()
    total, n = 0.0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == 'cuda'):
            logits = model(x)
        loss = loss_fn(logits.float(), y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total += loss.item() * x.size(0)
        n += x.size(0)
    return total / n


@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    m = WaterMetrics()
    total, n = 0.0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == 'cuda'):
            logits = model(x)
        total += loss_fn(logits.float(), y).item() * x.size(0)
        n += x.size(0)
        m.update((logits[:, 0] > 0).cpu().numpy(), y.cpu().numpy())
    return total / n, m

In [ ]:
def run_experiment(name, loss_name='bce', epochs=40, lr=1e-3, batch_size=8, crop=256,
                   train_splits=('train',), weak_per_epoch=0, init_from=None, force=False):
    """Entrena y guarda en outputs/<name>/.
    - Si el experimento ya existe en Drive, NO reentrena: lee su history.csv (force=True para reentrenar).
    - weak_per_epoch: cuántos chips débiles (elegidos al azar) se suman a los dorados en cada época.
    - init_from: nombre de otro experimento cuyo best.pt se usa como punto de partida (E5)."""
    exp_dir = f'{OUT}/{name}'
    if not force and os.path.exists(f'{exp_dir}/history.csv'):
        print(f'{name}: ya entrenado → se lee {exp_dir}/history.csv  (force=True para reentrenar)')
        return pd.read_csv(f'{exp_dir}/history.csv')
    os.makedirs(exp_dir, exist_ok=True)
    torch.manual_seed(SEED)
    rng = np.random.default_rng(SEED)

    gold_ids = chips.loc[chips['split'].isin(train_splits), 'chip'].tolist()
    valid_loader = DataLoader(FloodDataset(chips[chips['split'] == 'valid']),
                              batch_size=4, shuffle=False, num_workers=2)

    model = UNet().to(DEVICE)
    if init_from:
        model.load_state_dict(torch.load(f'{OUT}/{init_from}/best.pt', map_location=DEVICE)['model'])
        print(f'Pesos iniciales: {init_from}/best.pt')
    loss_fn = LOSSES[loss_name]
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')
    config = {'loss': loss_name, 'lr': lr, 'epochs': epochs, 'crop': crop, 'train_splits': list(train_splits),
              'weak_per_epoch': weak_per_epoch, 'init_from': init_from}

    history, best_iou = [], -1.0
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        ids = gold_ids + (rng.choice(weak['chip'], weak_per_epoch, replace=False).tolist() if weak_per_epoch else [])
        train_loader = DataLoader(FloodDataset(ids, train=True, crop=crop),
                                  batch_size=batch_size, shuffle=True, num_workers=2, drop_last=True)
        train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, scaler)
        val_loss, val_m = evaluate(model, valid_loader, loss_fn)
        scheduler.step()
        met = val_m.compute()
        history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss, **met})
        mejora = met['iou'] > best_iou
        if mejora:
            best_iou = met['iou']
            torch.save({'epoch': epoch, 'model': model.state_dict(), 'config': config}, f'{exp_dir}/best.pt')
        print(f"{epoch:3d} | train {train_loss:.4f} | val {val_loss:.4f} | IoU {met['iou']:.4f} | "
              f"F1 {met['f1']:.4f} | {time.time() - t0:.0f}s {'✓ mejor' if mejora else ''}")

    history = pd.DataFrame(history)
    history.to_csv(f'{exp_dir}/history.csv', index=False)
    print(f'\nMejor IoU en valid: {best_iou:.4f} → {exp_dir}/best.pt')
    return history

## 10. E1 — U-Net + BCE (modelo base)

40 épocas con recortes de 256: en una T4 debería tomar ≈ 10–15 minutos.

In [ ]:
hist_e1 = run_experiment('E1_bce', loss_name='bce')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(hist_e1['epoch'], hist_e1['train_loss'], label='train')
ax1.plot(hist_e1['epoch'], hist_e1['val_loss'], label='valid')
ax1.set_title('Loss'); ax1.set_xlabel('época'); ax1.legend(); ax1.grid(alpha=.3)

ax2.plot(hist_e1['epoch'], hist_e1['iou'], label='IoU')
ax2.plot(hist_e1['epoch'], hist_e1['f1'], label='F1')
ax2.axhline(otsu_valid.compute()['iou'], color='gray', ls='--', label='IoU Otsu')
ax2.set_title('Métricas en valid (clase agua)'); ax2.set_xlabel('época'); ax2.legend(); ax2.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 11. Comparación con Otsu (valid)

Recargamos el **mejor checkpoint** de E1 (no el de la última época) y lo comparamos con la línea base.

In [ ]:
def load_model(name):
    ckpt = torch.load(f'{OUT}/{name}/best.pt', map_location=DEVICE)
    model = UNet().to(DEVICE)
    model.load_state_dict(ckpt['model'])
    print(f"{name}: mejor época {ckpt['epoch']} · {ckpt['config']}")
    return model.eval()


valid_loader = DataLoader(FloodDataset(chips[chips['split'] == 'valid']), batch_size=4)
model_e1 = load_model('E1_bce')
_, e1_valid = evaluate(model_e1, valid_loader, masked_bce)

resultados = pd.DataFrame({'Otsu': otsu_valid.compute(), 'E1 · U-Net + BCE': e1_valid.compute()}).T.round(4)
resultados.to_csv(f'{OUT}/resultados_valid.csv')
resultados

In [ ]:
e1_valid.confusion()

## 12. Ejemplos visuales (valid)

In [ ]:
@torch.no_grad()
def predict(model, chip):
    img, _ = CACHE[chip]
    x = torch.from_numpy(normalize(img))[None].to(DEVICE)
    return (model(x)[0, 0] > 0).cpu().numpy()


ejemplos = chips[chips['split'] == 'valid'].sample(4, random_state=1)['chip']
fig, axes = plt.subplots(4, 4, figsize=(16, 16))
for row, chip in zip(axes, ejemplos):
    img, lab = CACHE[chip]
    with rasterio.open(f'{DATA}/S1OtsuLabelHand/{chip}_S1OtsuLabelHand.tif') as src:
        otsu = src.read(1)
    pred = predict(model_e1, chip)
    row[0].imshow(img[0], cmap='gray', vmin=-25, vmax=0); row[0].set_title(f'{chip} · VV'); row[0].axis('off')
    show_mask(row[1], lab, 'Real (LabelHand)')
    show_mask(row[2], np.where(lab == -1, -1, otsu == 1), 'Otsu')
    show_mask(row[3], np.where(lab == -1, -1, pred), 'U-Net E1')
plt.tight_layout(); plt.show()

## Criterio para cerrar la Fase 2 ✅

- [x] La curva de loss de train baja y la de valid no se dispara (train ≈ valid ≈ 0.14 al final: **no hay sobreajuste**).
- [x] **E1 supera a Otsu en IoU de valid: 0.613 frente a 0.507** (+0.106).
- [x] La prueba de la máscara (sección 6) da valores iguales en las tres losses (bce 0.1269, dice 0.0795, bce_dice 0.1032).

### Lectura de E1 (para el informe)

| | Precisión | Recall | Comisión (falsa agua) | Omisión (agua no vista) |
|---|---|---|---|---|
| Otsu | 0.63 | 0.72 | **0.37** | 0.28 |
| E1 | **0.83** | 0.70 | 0.17 | **0.30** |

- **Otsu marca agua de más:** confunde con agua todo lo oscuro, por ejemplo campos de cultivo lisos (`Paraguay_76868`) o sombras (`Paraguay_172476`).
- **E1 es conservadora:** casi no inventa agua (su comisión es la mitad), pero sigue dejando sin detectar un 30 % del agua real. Esto se ve sobre todo en zonas de agua "difusa" o con vegetación (`Paraguay_305760`).
- El **pico de la loss de valid en la época 1** es normal: la BatchNorm todavía no tiene buenas estadísticas.
- Las métricas oscilan hasta la época 15 y se estabilizan cuando baja el *learning rate* (coseno).
- **Hipótesis para la Fase 3:** la Dice no se deja dominar por el 90 % de "no agua". Por eso debería **subir el recall** (bajar la omisión), quizá a costa de algo de precisión.

## 13. Fase 3: ablación de la loss (E2 y E3)

Misma U-Net, mismos datos y mismas épocas que E1. **Solo cambia la loss**: así, cualquier diferencia se debe a ella.

| Exp | Loss |
|---|---|
| E1 | BCE (ya entrenado) |
| E2 | Dice |
| E3 | 0.5 · BCE + 0.5 · Dice |

Cada una tarda lo mismo que E1. Si Colab se desconecta, no pasa nada: cada experimento queda guardado en `outputs/<nombre>/`, y las celdas de comparación leen desde Drive.

In [ ]:
hist_e2 = run_experiment('E2_dice', loss_name='dice')

In [ ]:
hist_e3 = run_experiment('E3_bce_dice', loss_name='bce_dice')

### Curvas de las tres losses

Se leen los `history.csv` desde Drive. El valor de la loss **no se compara** entre experimentos, porque cada loss tiene su propia escala. Lo que se compara es el **IoU**.

In [ ]:
EXPS = {'E1_bce': 'E1 · BCE', 'E2_dice': 'E2 · Dice', 'E3_bce_dice': 'E3 · BCE+Dice'}
HIST = {name: pd.read_csv(f'{OUT}/{name}/history.csv') for name in EXPS}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
for name, label in EXPS.items():
    ax1.plot(HIST[name]['epoch'], HIST[name]['iou'], label=label)
    ax2.plot(HIST[name]['epoch'], HIST[name]['recall'], label=label)
for ax, t in [(ax1, 'IoU en valid'), (ax2, 'Recall en valid')]:
    ax.axhline(otsu_valid.compute()[t.split()[0].lower()], color='gray', ls='--', label='Otsu')
    ax.set_title(t); ax.set_xlabel('época'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

### Tabla comparativa (valid)

Dos columnas extra, para no engañarnos con un solo número:
- `mejor_epoca`: en qué época salió el mejor checkpoint.
- `iou_ult5`: el promedio del IoU en las últimas 5 épocas. El IoU de E1 oscila unos ±0.02 de una época a otra, así que **una diferencia menor que eso entre losses no es concluyente**.

In [ ]:
filas = {'Otsu': otsu_valid.compute()}
extra = {}
for name, label in EXPS.items():
    m = load_model(name)
    _, met = evaluate(m, valid_loader, masked_bce)
    filas[label] = met.compute()
    h = HIST[name]
    extra[label] = {'mejor_epoca': int(h.loc[h['iou'].idxmax(), 'epoch']), 'iou_ult5': h['iou'].tail(5).mean()}

resultados = pd.DataFrame(filas).T.join(pd.DataFrame(extra).T)
resultados.to_csv(f'{OUT}/resultados_valid.csv')
resultados.round(4)

### Las tres losses sobre los mismos chips

Son los mismos 4 chips de la sección 12, para comparar a ojo dónde mejora o empeora cada loss.

In [ ]:
modelos = {label: load_model(name) for name, label in EXPS.items()}

fig, axes = plt.subplots(4, 5, figsize=(20, 16))
for row, chip in zip(axes, ejemplos):
    img, lab = CACHE[chip]
    row[0].imshow(img[0], cmap='gray', vmin=-25, vmax=0); row[0].set_title(f'{chip} · VV'); row[0].axis('off')
    show_mask(row[1], lab, 'Real (LabelHand)')
    for ax, (label, m) in zip(row[2:], modelos.items()):
        show_mask(ax, np.where(lab == -1, -1, predict(m, chip)), label)
plt.tight_layout(); plt.show()

## Criterio para cerrar la Fase 3a (loss) ✅

- Se elige la loss con **mayor IoU en valid**. Si dos quedan a menos de ~0.02 (mirar también `iou_ult5`), se prefiere la de **mayor recall**: en inundaciones, no ver agua real es peor que marcar un poco de más.
- Esa loss se usa en **E4** (dorados + débiles) y **E5** (preentrenar con débiles).
- Anotar para el informe: la precisión y el recall de cada loss, no solo el IoU.

**Resultado:** E1 0.623 · E2 0.640 · E3 **0.643** de IoU. E2 y E3 empatan (la misma E1 varió 0.613 → 0.623 entre dos corridas), pero E3 tiene el mayor recall (0.778). → **Se usa `bce_dice` en E4 y E5.**

# Fase 3b: ¿ayudan las etiquetas débiles? (E4 y E5)

Hasta aquí entrenamos con **250 chips dorados** (etiquetados a mano). Sen1Floods11 trae además **4,384 chips débiles**: imágenes Sentinel-1 iguales a las nuestras, cuya máscara la hizo **un umbral automático (Otsu)**, no una persona.

| | Dorados | Débiles |
|---|---|---|
| Cantidad | 250 (train) | 4,160 (sin Bolivia) |
| Etiqueta | A mano: confiable | Umbral Otsu: con errores |
| Sin dato (−1) | Sí | No (solo donde la imagen es NaN) |

**La pregunta:** ¿17 veces más datos compensan que las etiquetas sean peores? En valid, Otsu tenía **comisión 0.37** (marca agua de más). La red podría aprender ese mismo error.

**Reglas:**
- **Bolivia se excluye**: es nuestro conjunto de generalización y no puede verse al entrenar.
- Se excluye también cualquier chip débil que **se superponga en el mapa** con un chip de valid, test o Bolivia (sería hacer trampa).
- Se usa la loss ganadora de la Fase 3a: **`bce_dice`**.

## 14. Descargar los chips débiles

Son ≈ 7 GB. Se descargan al **disco local de Colab** (`/content/weak`), **no a Drive**:
- Desde el bucket de Google a Colab tarda ≈ 3–5 min. Leer 4,000 archivos desde Drive sería mucho más lento.
- No ocupa espacio en tu Drive.
- **Se borra al cerrar la sesión.** Por eso esta celda se vuelve a ejecutar en cada sesión. `rsync` solo baja lo que falta: si ya está todo, termina en segundos.

In [ ]:
for sub in ['S1OtsuLabelWeak', 'S1Weak']:
    os.makedirs(f'{WEAK}/{sub}', exist_ok=True)
    !gsutil -m -q rsync -x "Bolivia_.*" gs://sen1floods11/v1.1/data/flood_events/WeaklyLabeled/{sub} "{WEAK}/{sub}"
    print(sub, len(os.listdir(f'{WEAK}/{sub}')), 'archivos (esperado 4160)')

In [ ]:
weak = pd.DataFrame({'chip': sorted(f.replace('_S1Weak.tif', '') for f in os.listdir(f'{WEAK}/S1Weak'))})
weak['evento'] = weak['chip'].str.split('_').str[0]
weak['evento'].value_counts()      # Mekong ≈ 1,350, Colombia 534 (no tiene chips dorados), ... sin Bolivia

### ¿Cuánta agua marcan las etiquetas débiles?

En los dorados, el agua es ≈ 10.6 % de los píxeles. Si las débiles marcan mucho más, es una señal de que el umbral exagera.

In [ ]:
def pct_agua_weak(chip):
    with rasterio.open(f'{WEAK}/S1OtsuLabelWeak/{chip}_S1OtsuLabelWeak.tif') as src:
        return 100 * (src.read(1) == 1).mean()

weak['pct_agua'] = [pct_agua_weak(c) for c in tqdm(weak['chip'])]
weak.groupby('evento')['pct_agua'].describe()[['count', 'mean', '50%', 'max']].round(1)

### Chips débiles que se superponen con valid / test / Bolivia

Los nombres de los chips débiles no coinciden con los dorados, pero **dos chips distintos pueden cubrir el mismo terreno**. Comparamos los rectángulos geográficos (`bounds`: izquierda, abajo, derecha, arriba) y quitamos los débiles que compartan más del 1 % de su área con un chip de evaluación.

In [ ]:
def bounds(path):
    with rasterio.open(path) as src:
        return tuple(src.bounds)

evaluacion = chips[chips['split'].isin(['valid', 'test', 'bolivia'])]
gb = np.array([bounds(f'{DATA}/S1Hand/{f}') for f in tqdm(evaluacion['s1_file'])])
wb = np.array([bounds(f'{WEAK}/S1Weak/{c}_S1Weak.tif') for c in tqdm(weak['chip'])])

# Ancho y alto del rectángulo en común entre cada chip débil (filas) y cada chip de evaluación (columnas)
ancho = np.minimum(wb[:, None, 2], gb[None, :, 2]) - np.maximum(wb[:, None, 0], gb[None, :, 0])
alto  = np.minimum(wb[:, None, 3], gb[None, :, 3]) - np.maximum(wb[:, None, 1], gb[None, :, 1])
comun = np.clip(ancho, 0, None) * np.clip(alto, 0, None)
weak['solape_eval'] = comun.max(axis=1) / ((wb[:, 2] - wb[:, 0]) * (wb[:, 3] - wb[:, 1]))

print('Chips débiles con solape > 1 % con valid/test/bolivia:', (weak['solape_eval'] > 0.01).sum())
weak = weak[weak['solape_eval'] <= 0.01].reset_index(drop=True)
print('Chips débiles disponibles para entrenar:', len(weak))

### Un vistazo a 3 chips débiles

Compara la máscara débil con la imagen. Fíjate si marca como agua zonas oscuras que no lo parecen: es el mismo error de Otsu que vimos en `Paraguay_76868`.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for row, chip in zip(axes, weak.sample(3, random_state=SEED)['chip']):
    img, lab = get_chip(chip)
    row[0].imshow(img[0], cmap='gray', vmin=-25, vmax=0);  row[0].set_title(f'{chip} · VV'); row[0].axis('off')
    row[1].imshow(img[1], cmap='gray', vmin=-30, vmax=-5); row[1].set_title('VH'); row[1].axis('off')
    show_mask(row[2], lab, 'Máscara débil (Otsu)')
plt.tight_layout(); plt.show()

## 15. E4: dorados + débiles mezclados

En cada época entran **los 250 dorados + 500 débiles elegidos al azar** (otros 500 en la época siguiente).
- Así, los dorados siguen pesando un tercio de cada época. Si se metieran los 4,000 débiles de una vez, taparían a los dorados, y además cada época tardaría 17 veces más.
- En 40 épocas, la red ve cada chip débil unas 5 veces.
- Tarda ≈ **3 veces lo que tardó E1**: mira los segundos por época en las primeras líneas.

In [ ]:
hist_e4 = run_experiment('E4_mix', loss_name='bce_dice', weak_per_epoch=500)

## 16. E5: preentrenar con débiles → ajustar con dorados

Dos pasos:
1. **Preentrenar (E5a):** 20 épocas **solo con débiles**, 1,000 al azar por época.
   - La red aprende "qué parece agua" con muchos ejemplos imperfectos.
   - Su IoU en valid, por sí solo, es interesante: muestra **cuánto rinde una U-Net que solo vio etiquetas de Otsu** comparada con el propio Otsu.
2. **Ajustar (E5):** se parte de esos pesos y se entrena 40 épocas **solo con dorados**, con un *learning rate* más bajo (3e-4), para corregir lo aprendido sin borrarlo.

Es la misma idea del *transfer learning*: primero cantidad, después calidad.

In [ ]:
hist_e5a = run_experiment('E5a_pre_debiles', loss_name='bce_dice', epochs=20, train_splits=(), weak_per_epoch=1000)

In [ ]:
hist_e5 = run_experiment('E5_pre_ajuste', loss_name='bce_dice', lr=3e-4, init_from='E5a_pre_debiles')

## 17. Comparación de todos los experimentos (valid)

Solo aparecen los experimentos que ya están en Drive, así que esta celda se puede ejecutar aunque falte alguno.

In [ ]:
EXPS_ALL = {'E1_bce': 'E1 · BCE', 'E2_dice': 'E2 · Dice', 'E3_bce_dice': 'E3 · BCE+Dice',
            'E4_mix': 'E4 · dorados+débiles', 'E5a_pre_debiles': 'E5a · solo débiles',
            'E5_pre_ajuste': 'E5 · pre débiles → dorados'}
EXPS_ALL = {k: v for k, v in EXPS_ALL.items() if os.path.exists(f'{OUT}/{k}/history.csv')}
HIST = {name: pd.read_csv(f'{OUT}/{name}/history.csv') for name in EXPS_ALL}

filas, extra = {'Otsu': otsu_valid.compute()}, {}
for name, label in EXPS_ALL.items():
    _, met = evaluate(load_model(name), valid_loader, masked_bce)
    filas[label] = met.compute()
    h = HIST[name]
    extra[label] = {'mejor_epoca': int(h.loc[h['iou'].idxmax(), 'epoch']), 'iou_ult5': h['iou'].tail(5).mean()}

resultados = pd.DataFrame(filas).T.join(pd.DataFrame(extra).T)
resultados.to_csv(f'{OUT}/resultados_valid.csv')
resultados.round(4)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
for name in ['E3_bce_dice', 'E4_mix', 'E5_pre_ajuste']:
    if name in HIST:
        ax1.plot(HIST[name]['epoch'], HIST[name]['iou'], label=EXPS_ALL[name])
        ax2.plot(HIST[name]['epoch'], HIST[name]['recall'], label=EXPS_ALL[name])
for ax, t in [(ax1, 'IoU en valid'), (ax2, 'Recall en valid')]:
    ax.axhline(otsu_valid.compute()[t.split()[0].lower()], color='gray', ls='--', label='Otsu')
    ax.set_title(t); ax.set_xlabel('época'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
modelos = {EXPS_ALL[n]: load_model(n) for n in ['E3_bce_dice', 'E4_mix', 'E5_pre_ajuste'] if n in EXPS_ALL}

fig, axes = plt.subplots(4, 2 + len(modelos), figsize=(4 * (2 + len(modelos)), 16))
for row, chip in zip(axes, ejemplos):
    img, lab = CACHE[chip]
    row[0].imshow(img[0], cmap='gray', vmin=-25, vmax=0); row[0].set_title(f'{chip} · VV'); row[0].axis('off')
    show_mask(row[1], lab, 'Real (LabelHand)')
    for ax, (label, m) in zip(row[2:], modelos.items()):
        show_mask(ax, np.where(lab == -1, -1, predict(m, chip)), label)
plt.tight_layout(); plt.show()

## Criterio para cerrar la Fase 3b ✅

- Se compara **E4 y E5 contra E3** (misma loss, solo dorados).
  - Si superan a E3 por más de ~0.02 de IoU, las etiquetas débiles **sí ayudan**.
  - Si no, también es un resultado válido para el informe: "más datos con etiquetas ruidosas no compensan".
- Mirar la **comisión**: si sube mucho, la red heredó el error de Otsu (marcar agua de más).
- El experimento con **mayor IoU en valid** pasa a ser el **modelo final**. Recién ese se evalúa en **test, Bolivia y Lima** (Fase 5).

### Resultado (valid)

| | IoU | Precisión | Recall | Comisión | iou_ult5 | mejor época |
|---|---|---|---|---|---|---|
| Otsu | 0.507 | 0.634 | 0.716 | 0.366 | – | – |
| **E3 · BCE+Dice (solo dorados)** | **0.643** | 0.787 | **0.778** | 0.213 | **0.618** | 30 |
| E4 · dorados + débiles | 0.577 | 0.779 | 0.690 | 0.221 | 0.567 | 13 |
| E5a · solo débiles | 0.548 | 0.768 | 0.657 | 0.232 | 0.542 | 10 |
| E5 · pre débiles → dorados | 0.620 | 0.792 | 0.741 | 0.208 | 0.601 | 23 |

**Las etiquetas débiles no ayudan.** E4 queda 0.066 por debajo de E3 y E5 queda 0.022 por debajo (su `iou_ult5` también es menor). → **Modelo final: E3 · BCE+Dice.**

Por qué (lo que se ve en las figuras):
- **Las máscaras débiles son "manchas suavizadas":** no tienen ríos finos ni charcos pequeños. E4 aprendió ese estilo: en `Paraguay_172476` **no ve el río** y en `Mekong_98310` pierde las manchas pequeñas → baja el recall (0.69).
- **Heredó el error de Otsu:** en `Paraguay_76868`, E4 marca como agua los **campos de cultivo lisos** de abajo, con manchas más grandes que E3.
- **Las débiles marcan más agua:** el promedio por chip va de 4.6 % (USA) a 30.5 % (Pakistan), frente a ≈ 10.6 % en los dorados. Mekong, el evento con más chips débiles (1,353), marca 21 %.
- E4 se estanca (≈ 0.567 desde la época 25): con dos tercios de cada época en etiquetas ruidosas, la red no llega al nivel de E3.
- E5 queda cerca de E3: el ajuste con dorados **corrige casi todo** lo aprendido de las débiles, pero no lo supera.

**Dato interesante para el informe:** E5a (una U-Net que **solo** vio etiquetas de Otsu) supera al propio Otsu: IoU 0.548 frente a 0.507, y su comisión baja de 0.37 a 0.23. La red no copia el umbral píxel a píxel: aprende la forma y el contexto del agua, y así filtra parte del ruido de sus propias etiquetas.

**Limitación:** cada experimento se corrió una sola vez (variación entre corridas ≈ 0.01), así que la diferencia de E5 frente a E3 (0.022) está en el límite. La de E4 (0.066) sí es clara.